# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laspric/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Answer:** One row represents one content item for one client on one reporting date.

For this assignment, I will use the March 2026 partition (`month=2026-03`) as my development and verification time window. I use a mid-panel month rather than the final June 2026 month so that the final month remains a sealed test/outcome window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature

- `gsc_impressions` — number of Search Console impressions.
- `gsc_clicks` — number of Search Console clicks.
- `ctr` — calculated as `gsc_clicks / gsc_impressions`.
- `gsc_avg_position` — Search Console average position.
- `ga4_engaged_sessions` — number of engaged Analytics sessions.

### Label

- `is_declining_label` — the target we want to predict.

### Context

- `content_hash_id` — identifies the content item and can be used for grouping/joining, but is not a model feature.
- `client_hash_id` — identifies the client and can be used for grouping/splitting, but is not a model feature.
- `report_date` — identifies the reporting date and helps define the time window.
- `gsc_data_available` — indicates whether Search Console data is available for the row.
- `ga4_data_available` — indicates whether Analytics data is available for the row.

### Excluded

- `trend_pct` — excluded because it is used to derive `trend_direction`, which is used to derive `is_declining_label`. Using it as a feature would leak information about the target.
- `trend_direction` — excluded because the label is derived from it, so it directly leaks information about the target.
- `content_hash_id` and `client_hash_id` — excluded from model features because they are identifiers rather than predictive measurements.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [10]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_keys,
        COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

grain_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────┐
│ total_rows │ distinct_grain_keys │ duplicate_rows │
│   int64    │        int64        │     int64      │
├────────────┼─────────────────────┼────────────────┤
│    9841378 │             9841378 │              0 │
└────────────┴─────────────────────┴────────────────┘



In [11]:
date_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

date_check.show()

┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   9841378 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘



Availability verification: In the March 2026 slice, 3,611,061 rows have Search Console data available (gsc_data_available IS TRUE) and 413,966 rows have GA4 data available (ga4_data_available IS TRUE). This shows that data availability is not uniform across all rows.

In [12]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

availability_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



In [15]:
features = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,

        gsc_impressions,
        gsc_clicks,

        CASE
            WHEN gsc_impressions > 0
            THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
            ELSE NULL
        END AS ctr,

        gsc_avg_position,
        ga4_engaged_sessions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    LIMIT 100
""")

features.show()

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬───────────────────────┬────────────────────┬──────────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │          ctr          │  gsc_avg_position  │ ga4_engaged_sessions │
│    date     │         varchar         │         varchar          │      int64      │   int64    │        double         │       double       │        int64         │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼───────────────────────┼────────────────────┼──────────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │              20 │          0 │                   0.0 │               3.35 │                 NULL │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │               1 │          0 │                   0.0 │                0.0 │                

### Why these features are available at the decision moment

- **gsc_impressions** — knowable at the decision moment because Search Console impressions for the reporting period have already been recorded before making the prediction.
- **gsc_clicks** — knowable at the decision moment because Search Console clicks for the reporting period are already available in the daily performance data.
- **ctr** — knowable at the decision moment because it is calculated from `gsc_clicks` and `gsc_impressions`, which are available at the same decision point.
- **gsc_avg_position** — knowable at the decision moment because the Search Console average position has already been recorded for the reporting period.
- **ga4_engaged_sessions** — knowable at the decision moment because GA4 engaged-session data is used only when the corresponding GA4 data is available.

In [17]:
label_sources = con.sql("""
    SELECT *
    FROM glob(
        'hf://datasets/FlyRank/internship-warehouse/**/*.parquet'
    )
    LIMIT 20
""")

label_sources.show()

┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                  file                                                  │
│                                                varchar                                                 │
├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/dim_content.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet │
│ hf://datasets/FlyRank/internship-wa

In [1]:
import duckdb
from google.colab import userdata

In [7]:
from google.colab import userdata

HF_TOKEN = userdata.get("FlyRankToken")

print("New FlyRank token loaded:", HF_TOKEN is not None)

New FlyRank token loaded: True


In [3]:
con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [8]:
con.execute("""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("DuckDB is now configured with the new FlyRank token.")

DuckDB is now configured with the new FlyRank token.


In [9]:
rel = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    LIMIT 5
""")

test.show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: The warehouse has unbalanced history across clients. Some clients have much longer Search Console and Analytics histories than others. In addition, some early rows may be GSC-only, while GA4 data is not yet available. Therefore, the data cannot always provide a complete and comparable history for every client and content item.

We also need to be careful about window overlap, because some 90-day query-level data can overlap with the period used for the outcome/label. This means we should only use features that would genuinely be available at the decision moment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.